# Spectral bandwidth experiments in the thesis

This notebook produces the fixed-sample Perron profiles, grid-fineness and
lambda paths, repeated-sample histograms, and increasing-sample-size comparisons.
Section 2 retains the chapter's `figure_3_5` and `localized` profiles.

Source: [KDE chapter, Numerical Experiments](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex).


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

code_directory = next(
    (candidate.resolve()
     for ancestor in (Path.cwd(), *Path.cwd().parents)
     for candidate in (ancestor, ancestor / "Code" / "Code_kde")
     if (candidate / "kde_tools" / "pdf_export.py").is_file()),
    None,
)
if code_directory is None:
    raise FileNotFoundError("Run this notebook from inside the MASTERARBEIT project.")
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))

import importlib

import matplotlib.pyplot as plt                                                                     ### PLOTTING
import numpy as np
from kde_tools.fourier import (
    DEFAULT_FREQUENCY_CUTOFF, DEFAULT_FREQUENCY_INTERVALS,
    DEFAULT_FOURIER_HALF_WIDTH, DEFAULT_FOURIER_SPATIAL_POINTS,
    DEFAULT_FOURIER_GRID_TAG,
)
import plotly.graph_objects as go                                                                   ### PLOTTING
from plotly.subplots import make_subplots                                                           ### PLOTTING

from kde_tools.perron import (
    PERRON_SOLVER_VERSION,
    doob_mass_from_perron,
    perron_eigenpair_from_divergence,
    perron_peak_index,
    perron_vector_from_divergence,
)
from kde_tools.pdf_export import enable_automatic_pdf_saving                                        ### PLOTTING
from kde_tools.bandwidths import (
    bandwidth_grid_linear_in_psi_beta,
    gaussian_mise_oracle_bandwidth,
    nearest_bandwidth_grid_index,
)
from kde_tools.density import exact_oracle_ise_values
from kde_tools.distribution_runs import run_beta2_distribution
from kde_tools.spectral_sweeps import SPECTRAL_SWEEP_RUNNER_VERSION, run_l2_point
from kde_tools.fourier import (
    beta2_divergence_matrix_fourier,
    kde_family_fft,
)
import kde_tools.plotting_functions as kde_plotting_functions                                       ### PLOTTING
import kde_tools.similarities as kde_similarities

# Reload helpers that may have changed since this kernel was started.
importlib.reload(kde_plotting_functions)
importlib.reload(kde_similarities)
from kde_tools.plotting_functions import (                                                          ### PLOTTING
    plot_bandwidth_distributions,                                                                   ### PLOTTING
    plot_perron_vector,                                                                             ### PLOTTING
)                                                                                                   ### PLOTTING
from kde_tools.similarities import (
    beta2_divergence_matrix,
    first_order_perron_vector,
    population_gaussian_beta2_divergence_matrix,
)

np.set_printoptions(precision=5, suppress=True)                                                     ### REPORTING
seed = 20260818

project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "Code" / "Code_kde").is_dir()
        and (candidate / "Abschlussarbeit_2021" / "LaTeX").is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate the MASTERARBEIT project root.")

kde_thesis_figure_dir = (
    project_root / "Code" / "Code_kde" / "outputs" / "images" / "thesis"
)
kde_thesis_figure_dir.mkdir(parents=True, exist_ok=True)                                            ### PLOTTING

def save_thesis_plotly_figure(figure, filename: str) -> None:
    from plotly.io import kaleido                                                                   ### PLOTTING
    kaleido.scope.mathjax = None
    figure.write_image(str(kde_thesis_figure_dir / filename), scale=1)                              ### PLOTTING

pdf_output_directory = enable_automatic_pdf_saving(                                                 ### PLOTTING
    "spectral_scale_selection_kde"                                                                  ### PLOTTING
)                                                                                                   ### PLOTTING

print(f"Perron solver: {PERRON_SOLVER_VERSION}")

from kde_tools.fourier import FOURIER_L2_METHOD_VERSION
from kde_tools.fourier import gaussian_kde_l2_gram_fourier



## 1. Example stationary distribution on a grid linear in $\psi_\beta$

Choose the sample size, number of bandwidth candidates, bandwidth interval, grid parameter $\beta$, and similarity scales below. The bandwidth candidates are uniformly spaced in $\psi_\beta(h)$. For $\beta=2$, this means uniform spacing in $\log h$, so the grid is geometric. The same fixed sample and grid are used for four Perron curves at $\lambda\in\{1,10,100,1000\}$. Dots show the Perron-vector components at the grid bandwidths, and a dashed vertical line in the corresponding curve colour marks each maximum. The blue dashed line marks the Gaussian MISE-oracle bandwidth, and the purple dashed line marks the sample-specific ISE-minimizing grid bandwidth.


In [ ]:
stationary_example_seed = seed + 5
stationary_example_n = 200
stationary_example_N = 191
stationary_example_beta = 2.0
stationary_example_h_min = 0.01
stationary_example_h_max = 2.0
stationary_example_lambdas = np.array([1.0, 10.0, 100.0, 1000.0])
stationary_example_frequency_cutoff = DEFAULT_FREQUENCY_CUTOFF
stationary_example_frequency_intervals = DEFAULT_FREQUENCY_INTERVALS
stationary_example_half_width = np.pi * stationary_example_frequency_intervals / stationary_example_frequency_cutoff
stationary_example_spatial_points = 2 * stationary_example_frequency_intervals

if stationary_example_n < 1:
    raise ValueError("stationary_example_n must be positive.")
if stationary_example_N < 2:
    raise ValueError("stationary_example_N must be at least two.")
if not 0.0 < stationary_example_h_min < stationary_example_h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if np.any(stationary_example_lambdas <= 0.0):
    raise ValueError("stationary_example_lambdas must be positive.")
if stationary_example_half_width <= 0.0:
    raise ValueError("stationary_example_half_width must be positive.")
if (
    stationary_example_spatial_points < 2
    or stationary_example_spatial_points % 2
):
    raise ValueError(
        "stationary_example_spatial_points must be a positive even integer."
    )

stationary_example_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=stationary_example_h_min,
    h_max=stationary_example_h_max,
    beta=stationary_example_beta,
    grid_size=stationary_example_N,
)
stationary_example_rng = np.random.default_rng(stationary_example_seed)
stationary_example_samples = stationary_example_rng.normal(
    size=stationary_example_n
)
stationary_example_divergence = beta2_divergence_matrix_fourier(
    samples=stationary_example_samples,
    h_grid=stationary_example_grid,
    half_width=stationary_example_half_width,
    spatial_points=stationary_example_spatial_points,
)
stationary_example_similarities = {}
stationary_example_perron_vectors = {}
stationary_example_spectral_gaps = {}
stationary_example_peak_indices = {}
stationary_example_peak_bandwidths = {}

for current_lambda in stationary_example_lambdas:
    current_similarity = np.exp(
        -current_lambda * stationary_example_divergence
    )
    np.fill_diagonal(current_similarity, 1.0)
    current_perron_result = perron_eigenpair_from_divergence(
        stationary_example_divergence, current_lambda
    )
    current_perron = current_perron_result.vector
    current_peak_index = int(perron_peak_index(current_perron))

    stationary_example_similarities[current_lambda] = current_similarity
    stationary_example_perron_vectors[current_lambda] = current_perron
    stationary_example_spectral_gaps[current_lambda] = float(
        np.exp(-current_lambda * current_perron_result.minimum_divergence)
        * current_perron_result.spectral_gap
        / (1.0 + np.exp(-current_lambda * current_perron_result.minimum_divergence)
           * current_perron_result.eigenvalue)
    )
    stationary_example_peak_indices[current_lambda] = current_peak_index
    stationary_example_peak_bandwidths[current_lambda] = float(
        stationary_example_grid[current_peak_index]
    )

stationary_example_reference_lambda = float(
    stationary_example_lambdas[0]
)
stationary_example_similarity = stationary_example_similarities[
    stationary_example_reference_lambda
]
stationary_example_perron = stationary_example_perron_vectors[
    stationary_example_reference_lambda
]
stationary_example_mass = doob_mass_from_perron(
    stationary_example_perron
)
stationary_example_peak_index = stationary_example_peak_indices[
    stationary_example_reference_lambda
]
stationary_example_peak_h = stationary_example_peak_bandwidths[
    stationary_example_reference_lambda
]
stationary_example_spectral_gap = stationary_example_spectral_gaps[
    stationary_example_reference_lambda
]
stationary_example_oracle_h = gaussian_mise_oracle_bandwidth(
    stationary_example_n
)
stationary_example_ise_values = exact_oracle_ise_values(
    samples=stationary_example_samples,
    h_grid=stationary_example_grid,
    sigma2=1.0,
)
stationary_example_ise_index = int(
    np.argmin(stationary_example_ise_values)
)
stationary_example_ise_h = float(
    stationary_example_grid[stationary_example_ise_index]
)
stationary_example_target_roughness = 3.0 / (8.0 * np.sqrt(np.pi))
stationary_example_h_speed = (
    3.0
    / (
        32.0
        * np.sqrt(np.pi)
        * stationary_example_n
        * stationary_example_target_roughness
    )
) ** (1.0 / 5.0)
stationary_example_log_h_min = np.log10(stationary_example_h_min)
stationary_example_log_h_max = np.log10(stationary_example_h_max)
stationary_example_log_h_span = (
    stationary_example_log_h_max - stationary_example_log_h_min
)
stationary_example_h_speed_paper_x = (
    np.log10(stationary_example_h_speed)
    - stationary_example_log_h_min
) / stationary_example_log_h_span
stationary_example_h_speed_index = (
    stationary_example_h_speed_paper_x
    * (stationary_example_N - 1)
)

fig, ax = plt.subplots(figsize=(10.5, 5.5))                                                         ### PLOTTING
stationary_example_curve_colors = (
    "#E69F00",  # orange
    "#009E73",  # bluish green
    "#D55E00",  # vermillion
    "#222222",  # near black
)
if len(stationary_example_curve_colors) != len(
    stationary_example_lambdas
):
    raise ValueError("Provide one curve colour per lambda value.")

for current_lambda, current_color in zip(
    stationary_example_lambdas,
    stationary_example_curve_colors,
):
    current_perron = stationary_example_perron_vectors[current_lambda]
    ax.plot(                                                                                        ### PLOTTING
        stationary_example_grid,                                                                    ### PLOTTING
        current_perron,                                                                             ### PLOTTING
        color=current_color,                                                                        ### PLOTTING
        linewidth=2.0,                                                                              ### PLOTTING
        marker="o",                                                                                 ### PLOTTING
        markersize=3.5,                                                                             ### PLOTTING
        label=rf"$\lambda={current_lambda:g}$",                                                     ### PLOTTING
    )                                                                                               ### PLOTTING

ax.set_xscale("log")                                                                                ### PLOTTING
ax.set_xlabel("Bandwidth h")                                                                        ### PLOTTING
ax.set_ylabel("Perron-vector component")                                                            ### PLOTTING
ax.grid(alpha=0.3)                                                                                  ### PLOTTING
for current_lambda, current_color in zip(                                                           ### PLOTTING
    stationary_example_lambdas,                                                                     ### PLOTTING
    stationary_example_curve_colors,                                                                ### PLOTTING
):                                                                                                  ### PLOTTING
    ax.axvline(                                                                                     ### PLOTTING
        stationary_example_peak_bandwidths[current_lambda],                                         ### PLOTTING
        color=current_color,                                                                        ### PLOTTING
        linestyle="--",                                                                             ### PLOTTING
        linewidth=1.2,                                                                              ### PLOTTING
        alpha=0.75,                                                                                 ### PLOTTING
        label="_nolegend_",                                                                         ### PLOTTING
    )                                                                                               ### PLOTTING
ax.axvline(                                                                                         ### PLOTTING
    stationary_example_oracle_h,                                                                    ### PLOTTING
    color="tab:blue",                                                                               ### PLOTTING
    linestyle="--",                                                                                 ### PLOTTING
    label=f"Gaussian MISE oracle: h={stationary_example_oracle_h:.4g}",                             ### PLOTTING
)                                                                                                   ### PLOTTING
ax.axvline(                                                                                         ### PLOTTING
    stationary_example_ise_h,                                                                       ### PLOTTING
    color="tab:purple",                                                                             ### PLOTTING
    linestyle="--",                                                                                 ### PLOTTING
    label=f"sample ISE minimum: h={stationary_example_ise_h:.4g}",                                  ### PLOTTING
)                                                                                                   ### PLOTTING
ax.legend(ncol=2, fontsize=8.5)                                                                     ### PLOTTING
plt.tight_layout()                                                                                  ### PLOTTING
fig.savefig(                                                                                        ### PLOTTING
    kde_thesis_figure_dir / "stationary_distribution_plot_001.pdf",                                 ### PLOTTING
    dpi=100,                                                                                        ### PLOTTING
)                                                                                                   ### PLOTTING
plt.show()                                                                                          ### PLOTTING


### Similarity scale versus chosen bandwidth

For the same fixed sample as Figure 3.1, the plot below uses $N=201$ bandwidths linear in $\psi_\beta$, varies the similarity scale $\lambda$ in $S_{ij}=\exp(-\lambda D_{2,ij})$, and records the bandwidth at the maximum of the Doob stationary mass. The blue dashed horizontal line marks the Gaussian MISE-oracle bandwidth, and the purple dashed horizontal line marks the sample-specific ISE-minimizing grid bandwidth.

The two red dashed horizontal lines mark the bandwidths in the closest adjacent pair on this grid, found by minimizing the adjacent beta-2 divergences.


In [ ]:
stationary_example_lambda_values = np.geomspace(1e-2, 1e4, 101)
stationary_example_lambda_grid_N = 201
stationary_example_lambda_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=stationary_example_h_min,
    h_max=stationary_example_h_max,
    beta=stationary_example_beta,
    grid_size=stationary_example_lambda_grid_N,
)
stationary_example_log_lambda_min = np.log10(
    stationary_example_lambda_values[0]
)
stationary_example_log_lambda_max = np.log10(
    stationary_example_lambda_values[-1]
)
stationary_example_lambda_ise_values = exact_oracle_ise_values(
    samples=stationary_example_samples,
    h_grid=stationary_example_lambda_grid,
    sigma2=1.0,
)
stationary_example_lambda_ise_index = int(
    np.argmin(stationary_example_lambda_ise_values)
)
stationary_example_lambda_ise_h = float(
    stationary_example_lambda_grid[
        stationary_example_lambda_ise_index
    ]
)
stationary_example_lambda_divergence = beta2_divergence_matrix_fourier(
    samples=stationary_example_samples,
    h_grid=stationary_example_lambda_grid,
    half_width=stationary_example_half_width,
    spatial_points=stationary_example_spatial_points,
)
# Identify the closest adjacent pair on the same grid used for selection.
stationary_example_lambda_closest_pair_left_index = int(
    np.argmin(np.diag(stationary_example_lambda_divergence, k=1))
)
stationary_example_lambda_closest_pair_indices = (
    stationary_example_lambda_closest_pair_left_index,
    stationary_example_lambda_closest_pair_left_index + 1,
)
stationary_example_lambda_closest_pair_bandwidths = (
    stationary_example_lambda_grid[
        list(stationary_example_lambda_closest_pair_indices)
    ]
)

stationary_example_lambda_selected_indices = np.empty(
    len(stationary_example_lambda_values),
    dtype=int,
)

for index, current_lambda in enumerate(
    stationary_example_lambda_values
):
    current_perron = perron_vector_from_divergence(
        stationary_example_lambda_divergence, current_lambda
    )
    current_mass = doob_mass_from_perron(current_perron)
    stationary_example_lambda_selected_indices[index] = int(
        perron_peak_index(current_perron)
    )

stationary_example_lambda_selected_bandwidths = (
    stationary_example_lambda_grid[
        stationary_example_lambda_selected_indices
    ]
)

fig, ax = plt.subplots(figsize=(10.5, 5.5))                                                         ### PLOTTING
ax.step(                                                                                            ### PLOTTING
    stationary_example_lambda_values,                                                               ### PLOTTING
    stationary_example_lambda_selected_bandwidths,                                                  ### PLOTTING
    where="post",                                                                                   ### PLOTTING
    color="#222222",                                                                                ### PLOTTING
    linewidth=2.0,                                                                                  ### PLOTTING
    marker="o",                                                                                     ### PLOTTING
    markersize=3.5,                                                                                 ### PLOTTING
    label=(                                                                                         ### PLOTTING
        "spectral selection "                                                                       ### PLOTTING
        rf"($N={stationary_example_lambda_grid_N}$)"                                                ### PLOTTING
    ),                                                                                              ### PLOTTING
)                                                                                                   ### PLOTTING
ax.axhline(                                                                                         ### PLOTTING
    stationary_example_oracle_h,                                                                    ### PLOTTING
    color="tab:blue",                                                                               ### PLOTTING
    linestyle="--",                                                                                 ### PLOTTING
    linewidth=1.8,                                                                                  ### PLOTTING
    label=f"Gaussian MISE oracle: h={stationary_example_oracle_h:.4g}",                             ### PLOTTING
)                                                                                                   ### PLOTTING
ax.axhline(                                                                                         ### PLOTTING
    stationary_example_lambda_ise_h,                                                                ### PLOTTING
    color="tab:purple",                                                                             ### PLOTTING
    linestyle="--",                                                                                 ### PLOTTING
    linewidth=1.8,                                                                                  ### PLOTTING
    label=(                                                                                         ### PLOTTING
        "sample ISE minimum: "                                                                      ### PLOTTING
        f"h={stationary_example_lambda_ise_h:.4g}"                                                  ### PLOTTING
    ),                                                                                              ### PLOTTING
)                                                                                                   ### PLOTTING
for grid_index, bandwidth in zip(                                                                  ### PLOTTING
    stationary_example_lambda_closest_pair_indices,                                                 ### PLOTTING
    stationary_example_lambda_closest_pair_bandwidths,                                              ### PLOTTING
):                                                                                                  ### PLOTTING
    ax.axhline(                                                                                     ### PLOTTING
        bandwidth,                                                                                  ### PLOTTING
        color="tab:red",                                                                            ### PLOTTING
        linestyle="--",                                                                             ### PLOTTING
        linewidth=0.9,                                                                              ### PLOTTING
        zorder=1.8,                                                                                 ### PLOTTING
        label=(                                                                                     ### PLOTTING
            "closest pair: "                                                                        ### PLOTTING
            rf"$h_{{{grid_index + 1}}}={bandwidth:.6f}$"                                             ### PLOTTING
        ),                                                                                          ### PLOTTING
    )                                                                                               ### PLOTTING
ax.set_xscale("log")                                                                                ### PLOTTING
ax.set_yscale("log")                                                                                ### PLOTTING
ax.set_xlim(                                                                                        ### PLOTTING
    stationary_example_lambda_values[0],                                                            ### PLOTTING
    stationary_example_lambda_values[-1],                                                           ### PLOTTING
)                                                                                                   ### PLOTTING
ax.set_ylim(                                                                                        ### PLOTTING
    stationary_example_h_min,                                                                       ### PLOTTING
    stationary_example_h_max,                                                                       ### PLOTTING
)                                                                                                   ### PLOTTING
ax.set_xlabel(r"similarity scale $\lambda$")                                                        ### PLOTTING
ax.set_ylabel("chosen bandwidth h")                                                                 ### PLOTTING
ax.grid(alpha=0.3)                                                                                  ### PLOTTING
ax.legend(fontsize=8.5)                                                                             ### PLOTTING
plt.tight_layout()                                                                                  ### PLOTTING
fig.savefig(                                                                                        ### PLOTTING
    kde_thesis_figure_dir                                                                           ### PLOTTING
    / "chosen_bandwidth_vs_lambda_M1_fixed_sample.pdf",                                             ### PLOTTING
    dpi=100,                                                                                        ### PLOTTING
)                                                                                                   ### PLOTTING
plt.show()                                                                                          ### PLOTTING


### Grid-fineness sensitivity across similarity scales

This experiment reuses the same fixed sample and bandwidth interval while varying the number $N=11,16,\ldots,1001$ of bandwidth candidates linear in $\psi_\beta$. It plots $N$ against the bandwidth selected by the Perron-vector maximum for $\lambda\in\{1,10,100,1000\}$. The curve colours match Figure 3.1. Envelope fits are omitted so that the four grid-stability trajectories remain directly comparable. A thin grey vertical line at $N=201$ identifies the grid size used in Figure 3.3.


In [ ]:
stationary_example_grid_fineness_lambdas = (
    stationary_example_lambdas.copy()
)
stationary_example_grid_fineness_N_values = np.arange(
    11, 1002, 5, dtype=int
)
stationary_example_grid_fineness_tick_values = np.arange(
    101, 1002, 100, dtype=int
)
stationary_example_grid_fineness_selected_indices = np.empty(
    (
        len(stationary_example_grid_fineness_lambdas),
        len(stationary_example_grid_fineness_N_values),
    ),
    dtype=int,
)
stationary_example_grid_fineness_selected_bandwidths = np.empty(
    stationary_example_grid_fineness_selected_indices.shape,
    dtype=float,
)

for position, current_N in enumerate(
    stationary_example_grid_fineness_N_values
):
    current_grid = bandwidth_grid_linear_in_psi_beta(
        h_min=stationary_example_h_min,
        h_max=stationary_example_h_max,
        beta=stationary_example_beta,
        grid_size=int(current_N),
    )
    current_divergence = beta2_divergence_matrix_fourier(
        samples=stationary_example_samples,
        h_grid=current_grid,
        half_width=stationary_example_half_width,
        spatial_points=stationary_example_spatial_points,
    )

    for lambda_position, current_lambda in enumerate(
        stationary_example_grid_fineness_lambdas
    ):
        current_perron = perron_vector_from_divergence(
            current_divergence, current_lambda
        )
        current_stationary_mass = doob_mass_from_perron(current_perron)
        current_selected_index = int(
            perron_peak_index(current_perron)
        )
        stationary_example_grid_fineness_selected_indices[
            lambda_position, position
        ] = current_selected_index
        stationary_example_grid_fineness_selected_bandwidths[
            lambda_position, position
        ] = current_grid[current_selected_index]

    if (position + 1) % 20 == 0 or position + 1 == len(                                             ### REPORTING
        stationary_example_grid_fineness_N_values                                                   ### REPORTING
    ):                                                                                              ### REPORTING
        print(                                                                                      ### REPORTING
            f"completed {position + 1}/"                                                            ### REPORTING
            f"{len(stationary_example_grid_fineness_N_values)} grid sizes"                          ### REPORTING
        )                                                                                           ### REPORTING

fig, ax = plt.subplots(figsize=(10.5, 5.5))                                                         ### PLOTTING

for lambda_position, (current_lambda, current_color) in enumerate(                                  ### PLOTTING
    zip(                                                                                            ### PLOTTING
        stationary_example_grid_fineness_lambdas,                                                   ### PLOTTING
        stationary_example_curve_colors,                                                            ### PLOTTING
    )                                                                                               ### PLOTTING
):                                                                                                  ### PLOTTING
    ax.plot(                                                                                        ### PLOTTING
        stationary_example_grid_fineness_N_values,                                                  ### PLOTTING
        stationary_example_grid_fineness_selected_bandwidths[                                       ### PLOTTING
            lambda_position                                                                         ### PLOTTING
        ],                                                                                          ### PLOTTING
        color=current_color,                                                                        ### PLOTTING
        linewidth=1.8,                                                                              ### PLOTTING
        marker="o",                                                                                 ### PLOTTING
        markersize=2.8,                                                                             ### PLOTTING
        label=rf"$\lambda={current_lambda:g}$",                                                     ### PLOTTING
    )                                                                                               ### PLOTTING

ax.set_xlabel("number of bandwidth candidates N")                                                   ### PLOTTING
ax.set_ylabel("chosen bandwidth h")                                                                 ### PLOTTING
ax.set_xticks(stationary_example_grid_fineness_tick_values)                                         ### PLOTTING
ax.grid(alpha=0.3)                                                                                  ### PLOTTING
ax.axvline(                                                                                         ### PLOTTING
    stationary_example_lambda_grid_N,                                                               ### PLOTTING
    color="#7f7f7f",                                                                                ### PLOTTING
    linewidth=1.0,                                                                                  ### PLOTTING
    label=rf"Figure 3.3 uses $N={stationary_example_lambda_grid_N}$",                               ### PLOTTING
    zorder=0,                                                                                       ### PLOTTING
)                                                                                                   ### PLOTTING
ax.legend(ncol=3, fontsize=8.5)                                                                     ### PLOTTING
plt.tight_layout()                                                                                  ### PLOTTING
fig.savefig(                                                                                        ### PLOTTING
    kde_thesis_figure_dir / "grid_fineness_chosen_bandwidths.pdf",                                  ### PLOTTING
    dpi=100,                                                                                        ### PLOTTING
)                                                                                                   ### PLOTTING
plt.show()                                                                                          ### PLOTTING


## 2. Configurable beta-2 selection histogram

Choose `beta2_profile = "figure_3_5"` for the original lambda = 1 comparison, or
`beta2_profile = "localized"` for lambda = N^(3/2). Both profiles use the same
Gaussian samples and three grid selectors: spectral, ISE, and Q-hat.

The default n = 200, N = 1000, M = 1000 and seed reproduce the earlier
experiments. Both profiles now save resumable checkpoints, selections, histogram
frequencies, and per-repetition eigenpair residual/gap diagnostics. Runs are
separated by profile and complete configuration under
`outputs/data/spectral_scale_selection_kde/`; compatible earlier checkpoints
can be imported after checking settings, grid, sample hashes, and method versions.
Historical results remain in their original directories.

Run the next three cells after changing the profile. The localized profile
replaces the former standalone localized-selection notebook. The Figure 3.5
profile updates its thesis figure; the localized profile saves its own PNG and
automatic PDF. The following sample-size sweep has its own lambda setting.

Sources: [shared experiment runner](../kde_tools/distribution_runs.py);
[Perron solver and diagnostics](../kde_tools/perron.py);
[saved original localized settings](../outputs/data/spectral_bandwidth_distribution_n1000_lambda_n3_2_m1000/settings.json).


The `exact_gaussian` profile replaces the former Gaussian L²/ISE notebook. It uses its original independent seeds (`910000 + repetition`), exact pair divergence, and preview/thesis grids. It exports selected ISE values, an overlay, and the two individual distributions. Choose the profile in the next cell and rerun these three cells.


In [ ]:
beta2_profile = 'figure_3_5'
beta2_n = 200
beta2_N = 1000
beta2_M = 1000
beta2_binning = 5
beta2_h_min = 0.01
beta2_h_max = 2.0
if beta2_profile not in {'figure_3_5', 'localized'}:
    raise ValueError('Choose figure_3_5 or localized for beta2_profile.')
beta2_lambda = 1.0 if beta2_profile != 'localized' else beta2_N ** (3.0 / 2.0)
beta2_seed = seed + 10
beta2_sigma2 = 1.0
beta2_show_mise_reference = True
beta2_frequency_cutoff = DEFAULT_FREQUENCY_CUTOFF
beta2_frequency_intervals = DEFAULT_FREQUENCY_INTERVALS
beta2_half_width = np.pi * beta2_frequency_intervals / beta2_frequency_cutoff
beta2_spatial_points = 2 * beta2_frequency_intervals
beta2_validate_fft = True
beta2_workers = 4
beta2_reuse_checkpoints = True
beta2_data_root = code_directory / 'outputs/data/spectral_scale_selection_kde'
beta2_image_root = code_directory / 'outputs/images/spectral_scale_selection_kde'
beta2_image_directory = beta2_image_root / beta2_profile
if beta2_n < 1 or beta2_M < 1:
    raise ValueError('beta2_n and beta2_M must be positive.')
if beta2_binning < 1 or int(beta2_binning) != beta2_binning:
    raise ValueError('beta2_binning must be a positive integer.')
if beta2_N < 2:
    raise ValueError('beta2_N must be at least 2.')
if not 0.0 < beta2_h_min < beta2_h_max:
    raise ValueError('Require 0 < beta2_h_min < beta2_h_max.')
if beta2_lambda <= 0.0:
    raise ValueError('beta2_lambda must be positive.')
if beta2_half_width <= 0.0:
    raise ValueError('beta2_half_width must be positive.')
if beta2_spatial_points < 2:
    raise ValueError('beta2_spatial_points must be at least 2.')
beta2_h_grid = bandwidth_grid_linear_in_psi_beta(h_min=beta2_h_min, h_max=beta2_h_max, beta=2.0, grid_size=beta2_N)
print(f'beta=2, n={beta2_n}, N={beta2_N}, M={beta2_M}, lambda={beta2_lambda:g}')


In [ ]:
beta2_legacy_names = ['spectral_bandwidth_distribution_n1000_lambda_n3_2_m1000'] if beta2_profile == 'localized' else ['figure_3_5_bandwidth_distribution_n1000']
beta2_import_directories = [directory for name in beta2_legacy_names for directory in (code_directory / 'outputs/data' / name / FOURIER_L2_METHOD_VERSION, code_directory / 'outputs/data' / name)]
beta2_control_directories = [code_directory / 'outputs/data/figure_3_5_bandwidth_distribution_n1000' / FOURIER_L2_METHOD_VERSION, code_directory / 'outputs/data/figure_3_5_bandwidth_distribution_n1000']
beta2_run = run_beta2_distribution(beta2_h_grid, n=beta2_n, repetitions=beta2_M, lambda_value=beta2_lambda, seed=beta2_seed, data_root=beta2_data_root, profile=beta2_profile, half_width=beta2_half_width, spatial_points=beta2_spatial_points, workers=beta2_workers, reuse=beta2_reuse_checkpoints, import_directories=beta2_import_directories, control_directories=beta2_control_directories, validate_fourier=beta2_validate_fft, binning=beta2_binning, sigma2=beta2_sigma2)
beta2_selected_indices = beta2_run['selected_indices'][:, 0]
beta2_ise_selected_indices = beta2_run['selected_indices'][:, 1]
beta2_selection_counts, beta2_ise_selection_counts = beta2_run['counts'][:2]
beta2_qhat_selected_indices = beta2_run['selected_indices'][:, 2]
beta2_qhat_selection_counts = beta2_run['counts'][2]
beta2_solver_diagnostics = beta2_run['solver_diagnostics']
beta2_run_directory = beta2_run['run_directory']
beta2_image_directory = beta2_image_root / beta2_profile / beta2_run['configuration_id']
print(f'Saved selections and eigenpair diagnostics: {beta2_run_directory}')
print(f'Maximum relative residual: {beta2_solver_diagnostics[:, 0].max():.3e}')
print(f'Minimum relative spectral gap: {beta2_solver_diagnostics[:, 1].min():.3e}')


In [ ]:
from matplotlib.ticker import MultipleLocator, PercentFormatter
beta2_histogram_counts = np.stack([beta2_selection_counts, beta2_ise_selection_counts, beta2_qhat_selection_counts])
assert beta2_histogram_counts.shape == (3, beta2_N)
beta2_bin_starts = np.arange(0, beta2_N, beta2_binning)
beta2_bin_stops = np.minimum(beta2_bin_starts + beta2_binning, beta2_N)
beta2_bin_centers = (beta2_bin_starts + beta2_bin_stops - 1) / 2.0
beta2_bin_widths = beta2_bin_stops - beta2_bin_starts
beta2_binned_counts = np.add.reduceat(beta2_histogram_counts, beta2_bin_starts, axis=1)
assert np.array_equal(beta2_binned_counts.sum(axis=1), beta2_histogram_counts.sum(axis=1))
assert np.all(beta2_binned_counts.sum(axis=1) == beta2_M)
beta2_distribution_names = ['spectral selections', 'ISE-optimal grid bandwidths', 'Q-hat minimizers']
beta2_histogram, beta2_axis = plt.subplots(figsize=(12.0, 6.32))
beta2_histogram.subplots_adjust(left=0.085, right=0.98, top=0.88, bottom=0.25)
for beta2_label, beta2_color, beta2_opacity, beta2_values in zip(beta2_distribution_names, ['#C65D00', '#9467BD', '#66C2A5'], [0.7, 0.7, 0.55], beta2_binned_counts):
    beta2_nonzero = np.flatnonzero(beta2_values)
    beta2_axis.bar(beta2_bin_centers[beta2_nonzero], beta2_values[beta2_nonzero] / beta2_M, width=0.94 * beta2_bin_widths[beta2_nonzero], color=beta2_color, alpha=beta2_opacity, edgecolor='white', linewidth=0.3, label=beta2_label)
beta2_mise_bandwidth = gaussian_mise_oracle_bandwidth(beta2_n)
beta2_grid_indices = np.arange(beta2_N, dtype=float)
beta2_mise_grid_position = float(np.interp(np.log(beta2_mise_bandwidth), np.log(beta2_h_grid), beta2_grid_indices))
beta2_reference_label = f'Gaussian MISE optimum: h={beta2_mise_bandwidth:.4g}'
beta2_axis.axvline(beta2_mise_grid_position, color='#1f77b4', linestyle=(0, (5, 5)), linewidth=1.5, label=beta2_reference_label)
beta2_ticks = np.unique(np.linspace(0, beta2_N - 1, 7, dtype=int))
beta2_axis.set_xticks(beta2_ticks, [f'{beta2_h_grid[i]:.3g}' for i in beta2_ticks])
beta2_axis.set_xlim(-0.5, beta2_N - 0.5)
beta2_axis.set_ylim(0, 1.05 * float(beta2_binned_counts.max()) / beta2_M)
beta2_axis.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
beta2_axis.yaxis.set_major_locator(MultipleLocator(0.01))
beta2_axis.set_xlabel('selected bandwidth h', fontsize=12, labelpad=12, color='#304967')
beta2_axis.set_ylabel('relative frequency', fontsize=12, labelpad=12, color='#304967')
beta2_axis.tick_params(axis='both', labelsize=10, colors='#304967', length=0)
beta2_axis.grid(axis='y', color='#E5ECF6', linewidth=0.6)
beta2_axis.set_axisbelow(True)
for beta2_side in ('top', 'right', 'left'):
    beta2_axis.spines[beta2_side].set_visible(False)
beta2_axis.spines['bottom'].set_color('#E5ECF6')
beta2_histogram.suptitle(f'{('Figure 3.5' if beta2_profile == 'figure_3_5' else 'Localized spectral selection')}: {beta2_binning} consecutive grid points per bin\n' + f'$N={beta2_N}$, $n={beta2_n}$, $M={beta2_M}$, $\\lambda={beta2_lambda:g}$', y=0.965, fontsize=14, color='#304967')
beta2_handles, beta2_legend_labels = beta2_axis.get_legend_handles_labels()
beta2_legend_order = [beta2_legend_labels.index(label) for label in [*beta2_distribution_names, beta2_reference_label]]
beta2_histogram.legend([beta2_handles[i] for i in beta2_legend_order], [beta2_legend_labels[i] for i in beta2_legend_order], loc='lower center', bbox_to_anchor=(0.52, 0.035), ncol=2, frameon=False, fontsize=10, labelcolor='#304967')
beta2_image_directory.mkdir(parents=True, exist_ok=True)
beta2_png_path = beta2_image_directory / f'{beta2_profile}_N{beta2_N}_bins{beta2_binning}.png'
beta2_histogram.savefig(beta2_png_path, dpi=180, facecolor='white')
print(f'Saved comparison figure: {beta2_png_path}')
if beta2_profile == 'figure_3_5':
    beta2_thesis_figure_path = kde_thesis_figure_dir / 'distribution_of_chosen_bandwidths.pdf'
    beta2_histogram.savefig(beta2_thesis_figure_path, format='pdf', bbox_inches='tight')
    print(f'Updated Figure 3.5: {beta2_thesis_figure_path}')
print(f'N={beta2_N}; binning={beta2_binning}; bins={len(beta2_bin_starts)}')
plt.show()
plt.close(beta2_histogram)


The selected panels use n=300,400,500,1000 and retain the thesis delivery paths.
The all-size spectral overlays compare lambda=1 with lambda=N^(3/2), as in the chapter.


In [ ]:
import hashlib
import json
from concurrent.futures import ThreadPoolExecutor
from threadpoolctl import threadpool_limits

beta2_n_sweep_values = np.array([300, 400, 500, 1000], dtype=int)
beta2_n_sweep_N = 1000
beta2_n_sweep_binning = 5
beta2_n_sweep_workers = 4
beta2_n_sweep_M = 1000
beta2_n_sweep_h_min = beta2_h_min
beta2_n_sweep_h_max = beta2_h_max
beta2_n_sweep_lambda = 1.0  # Baseline, independent of the section 2 profile.
beta2_n_sweep_localized_lambda = beta2_n_sweep_N ** 1.5
beta2_n_sweep_lambda_values = [beta2_n_sweep_lambda, beta2_n_sweep_localized_lambda]
beta2_n_sweep_seed = seed + 12  # Keep this sample bank independent of section 2 profiles.
beta2_n_sweep_reuse_checkpoints = True
beta2_n_sweep_output_root = code_directory / "outputs/data/spectral_scale_selection_kde/sample_size_sweep"
beta2_n_sweep_cache_root = code_directory / "outputs/data/shared_spectral_sweeps"

beta2_n_sweep_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=beta2_n_sweep_h_min,
    h_max=beta2_n_sweep_h_max,
    beta=2.0,
    grid_size=beta2_n_sweep_N,
)
beta2_n_sweep_grid_indices = np.arange(
    beta2_n_sweep_N, dtype=float
)
beta2_n_sweep_oracle_bandwidths = {
    int(current_n): gaussian_mise_oracle_bandwidth(int(current_n))
    for current_n in beta2_n_sweep_values
}
beta2_n_sweep_oracle_indices = {
    int(current_n): float(np.interp(
        np.log(beta2_n_sweep_oracle_bandwidths[int(current_n)]),
        np.log(beta2_n_sweep_h_grid),
        beta2_n_sweep_grid_indices,
    ))
    for current_n in beta2_n_sweep_values
}
beta2_n_sweep_density_half_width = 12.0
beta2_n_sweep_density_spatial_points = 8192
beta2_n_sweep_dx = (
    2.0 * beta2_n_sweep_density_half_width / beta2_n_sweep_density_spatial_points
)
beta2_n_sweep_spatial_grid = (
    -beta2_n_sweep_density_half_width
    + beta2_n_sweep_dx * np.arange(beta2_n_sweep_density_spatial_points)
)
beta2_n_sweep_true_density = (
    np.exp(-0.5 * beta2_n_sweep_spatial_grid**2)
    / np.sqrt(2.0 * np.pi)
)
beta2_n_sweep_frequencies = 2.0 * np.pi * np.fft.rfftfreq(
    beta2_n_sweep_density_spatial_points, d=beta2_n_sweep_dx
)
beta2_n_sweep_parseval_weights = np.full(
    len(beta2_n_sweep_frequencies), 2.0
)
beta2_n_sweep_parseval_weights[0] = 1.0
beta2_n_sweep_parseval_weights[-1] = 1.0

beta2_n_sweep_rng = np.random.default_rng(beta2_n_sweep_seed)
beta2_n_sweep_full_samples = beta2_n_sweep_rng.normal(
    size=(beta2_n_sweep_M, int(np.max(beta2_n_sweep_values)))
)
beta2_n_sweep_settings = {
    "repetitions": beta2_n_sweep_M, "grid_size": beta2_n_sweep_N,
    "sigma2": 1.0, "h_min": beta2_n_sweep_h_min, "h_max": beta2_n_sweep_h_max,
    "fft_half_width": beta2_half_width, "fft_spatial_points": beta2_spatial_points,
    "seed": beta2_n_sweep_seed, "maximum_sample_size": int(np.max(beta2_n_sweep_values)),
}
beta2_n_sweep_configuration = {
    "settings": beta2_n_sweep_settings,
    "sample_sizes": [int(n) for n in beta2_n_sweep_values],
    "lambda_values": beta2_n_sweep_lambda_values, "spectral_sweep_runner": SPECTRAL_SWEEP_RUNNER_VERSION,
    "l2_method": FOURIER_L2_METHOD_VERSION, "perron_solver": PERRON_SOLVER_VERSION,
    "density_half_width": beta2_n_sweep_density_half_width,
    "density_spatial_points": beta2_n_sweep_density_spatial_points,
}
beta2_n_sweep_configuration_id = hashlib.sha256(
    json.dumps(beta2_n_sweep_configuration, sort_keys=True).encode()
).hexdigest()[:12]
beta2_n_sweep_output_directory = beta2_n_sweep_output_root / beta2_n_sweep_configuration_id
beta2_n_sweep_import_directories = [
    directory
    for profile in ("thesis", "preview")
    for directory in (code_directory / "outputs/data/fast_l2_bandwidth_distributions_vs_n_lambda" / profile).glob("*")
    if directory.is_dir()
]
beta2_n_sweep_spectral_results = {}
beta2_n_sweep_results = {}

def run_beta2_n_sweep_point(current_n):
    current_n = int(current_n)
    spectral_result = run_l2_point(
        current_n, beta2_n_sweep_full_samples, beta2_n_sweep_h_grid,
        beta2_n_sweep_settings, beta2_n_sweep_lambda_values, beta2_n_sweep_output_directory,
        reuse=beta2_n_sweep_reuse_checkpoints, cache_root=beta2_n_sweep_cache_root,
        import_directories=beta2_n_sweep_import_directories, diagnostics=True,
    )
    selected_indices = spectral_result["selected_indices"][:, 0]
    localized_selected_indices = spectral_result["selected_indices"][:, 1]
    ise_selected_indices = np.full(beta2_n_sweep_M, -1, dtype=int)
    qhat_selected_indices = np.full(beta2_n_sweep_M, -1, dtype=int)
    controls_path = spectral_result["point_directory"] / "grid_controls.npz"
    controls_identity = json.dumps({
        "samples_sha256": spectral_result["samples_sha256"],
        "grid": beta2_n_sweep_h_grid.tolist(),
        "half_width": beta2_n_sweep_density_half_width,
        "spatial_points": beta2_n_sweep_density_spatial_points,
        "method": "periodic_reconstructed_ise_log_derivative_q2_v1",
    }, sort_keys=True)
    controls_completed = 0
    controls_source = controls_path
    # The ISE/Q-hat controls do not depend on lambda. Import a fully completed
    # matching control checkpoint when adding another lambda configuration.
    if beta2_n_sweep_reuse_checkpoints and not controls_path.is_file():
        for candidate in sorted(beta2_n_sweep_output_root.glob(f"*/n{current_n}/grid_controls.npz")):
            with np.load(candidate, allow_pickle=False) as saved:
                if (str(saved["identity_json"].item()) == controls_identity
                        and int(saved["completed"].item()) == beta2_n_sweep_M):
                    controls_source = candidate
                    break
    if beta2_n_sweep_reuse_checkpoints and controls_source.is_file():
        with np.load(controls_source, allow_pickle=False) as saved:
            if str(saved["identity_json"].item()) != controls_identity:
                raise ValueError("Saved grid controls differ from the requested inputs.")
            ise_selected_indices = saved["ise_selected_indices"].copy()
            qhat_selected_indices = saved["qhat_selected_indices"].copy()
            controls_completed = int(saved["completed"].item())
        if (ise_selected_indices.shape != (beta2_n_sweep_M,)
                or qhat_selected_indices.shape != (beta2_n_sweep_M,)
                or not 0 <= controls_completed <= beta2_n_sweep_M):
            raise ValueError("Invalid control checkpoint dimensions or completion count.")
        for values in (ise_selected_indices, qhat_selected_indices):
            if (not np.issubdtype(values.dtype, np.integer)
                    or np.any(values[:controls_completed] < 0)
                    or np.any(values[:controls_completed] >= beta2_n_sweep_N)):
                raise ValueError("Invalid saved control selection indices.")
        print(f"n={current_n}: reused {controls_completed} ISE/Q-hat control pairs from {controls_source}", flush=True)

    controls_reused_count = controls_completed

    def save_controls():
        temporary = controls_path.with_suffix(".npz.tmp")
        with temporary.open("wb") as stream:
            np.savez_compressed(stream, identity_json=controls_identity,
                                ise_selected_indices=ise_selected_indices,
                                qhat_selected_indices=qhat_selected_indices,
                                completed=controls_completed)
        temporary.replace(controls_path)

    try:
        for repetition in range(controls_completed, beta2_n_sweep_M):
            current_samples = beta2_n_sweep_full_samples[repetition, :current_n]
            current_densities, current_dx = kde_family_fft(
                samples=current_samples, bandwidths=beta2_n_sweep_h_grid,
                half_width=beta2_n_sweep_density_half_width,
                spatial_points=beta2_n_sweep_density_spatial_points,
            )
            current_ise_values = current_dx * np.sum(
                (current_densities - beta2_n_sweep_true_density[None, :]) ** 2, axis=1,
            )
            ise_selected_indices[repetition] = int(np.argmin(current_ise_values))
            current_density_coefficients = np.fft.rfft(current_densities, axis=1)
            current_log_derivative_coefficients = (
                -beta2_n_sweep_h_grid[:, None] ** 2
                * beta2_n_sweep_frequencies[None, :] ** 2
                * current_density_coefficients
            )
            current_qhat_values = (
                .5 * current_dx / beta2_n_sweep_density_spatial_points
                * np.sum(beta2_n_sweep_parseval_weights[None, :]
                         * np.abs(current_log_derivative_coefficients) ** 2, axis=1)
            )
            qhat_selected_indices[repetition] = int(np.argmin(current_qhat_values))
            controls_completed = repetition + 1
            if controls_completed % 25 == 0 or controls_completed == beta2_n_sweep_M:
                save_controls()
            if controls_completed % 100 == 0:
                print(f"n={current_n}: grid ISE/Q-hat controls {controls_completed}/{beta2_n_sweep_M}", flush=True)
    finally:
        save_controls()

    result = {
        "selected_indices": selected_indices.copy(),
        "counts": np.bincount(selected_indices, minlength=beta2_n_sweep_N),
        "localized_selected_indices": localized_selected_indices.copy(),
        "localized_counts": np.bincount(localized_selected_indices, minlength=beta2_n_sweep_N),
        "controls_reused_from": str(controls_source) if controls_reused_count else None,
        "ise_selected_indices": ise_selected_indices.copy(),
        "ise_counts": np.bincount(ise_selected_indices, minlength=beta2_n_sweep_N),
        "qhat_selected_indices": qhat_selected_indices.copy(),
        "qhat_counts": np.bincount(qhat_selected_indices, minlength=beta2_n_sweep_N),
        "oracle_bandwidth": beta2_n_sweep_oracle_bandwidths[current_n],
        "oracle_index": beta2_n_sweep_oracle_indices[current_n],
    }
    result["relative_frequencies"] = result["counts"] / beta2_n_sweep_M
    assert all(result[key].sum() == beta2_n_sweep_M for key in ("counts", "ise_counts", "qhat_counts", "localized_counts"))
    print(f"completed n={current_n}", flush=True)
    return current_n, spectral_result, result


# Each sample size has independent checkpoints; limit numerical-library threads
# so the four workers do not oversubscribe the machine.
with threadpool_limits(limits=1):
    with ThreadPoolExecutor(max_workers=beta2_n_sweep_workers) as executor:
        for current_n, spectral_result, result in executor.map(run_beta2_n_sweep_point, beta2_n_sweep_values):
            beta2_n_sweep_spectral_results[current_n] = spectral_result
            beta2_n_sweep_results[current_n] = result


In [ ]:
beta2_n_individual_figures = {}

import kde_tools.bandwidth_histograms as kde_bandwidth_histograms
importlib.reload(kde_bandwidth_histograms)

from kde_tools.bandwidth_histograms import (
    bandwidth_distribution_names, plot_grouped_bandwidth_distributions,
)

for current_n in beta2_n_sweep_values:
    current_n = int(current_n)
    result = beta2_n_sweep_results[current_n]
    counts = np.stack([result[key] for key in ("counts", "ise_counts", "qhat_counts")])
    current_figure, _, bin_values = plot_grouped_bandwidth_distributions(
        beta2_n_sweep_h_grid, counts, sample_size=current_n,
        repetitions=beta2_n_sweep_M, lambda_value=beta2_n_sweep_lambda,
        positions_per_bin=beta2_n_sweep_binning,
        localized_counts=result["localized_counts"],
    )
    beta2_n_individual_figures[current_n] = current_figure
    if current_n in {300, 400, 500, 1000}:
        current_figure.savefig(
            kde_thesis_figure_dir / f"distribution_of_chosen_bandwidths_n{current_n}.pdf",
            format="pdf", bbox_inches="tight",
        )
    # Preserve the exact displayed bins beside the run's checkpoints.
    starts, stops, grouped = bin_values
    np.savetxt(
        beta2_n_sweep_output_directory / f"n{current_n}" / "grouped_histogram_counts.csv",
        np.column_stack((starts, stops - 1, grouped.T)), delimiter=",", fmt="%d",
        header="first_grid_index,last_grid_index,spectral_lambda1,ise,qhat,spectral_lambda_N3_2", comments="",
    )
    plt.show()  # Shared automatic exporter saves every displayed figure as PDF.
    plt.close(current_figure)

# Matching spectral overlays use one shared style for both lambda regimes.
from kde_tools.bandwidth_histograms import grouped_grid_counts


def plot_spectral_sample_size_overlay(*, count_key, plot_title, lambda_text,
                                      output_filename, csv_filename, tick_step):
    sample_overlay_sizes = [300, 400, 500, 1000]
    if all(n in beta2_n_sweep_results for n in sample_overlay_sizes):
        sample_overlay_counts = np.stack([
            beta2_n_sweep_results[n][count_key] for n in sample_overlay_sizes
        ])
        overlay_starts, overlay_stops, overlay_grouped = grouped_grid_counts(
            sample_overlay_counts, beta2_n_sweep_N, beta2_n_sweep_M,
            beta2_n_sweep_binning,
        )
        sample_overlay_figure, sample_overlay_axis = plt.subplots(figsize=(11.4, 6.6))
        sample_overlay_figure.subplots_adjust(left=.085, right=.98, top=.84, bottom=.21)
        overlay_centers = (overlay_starts + overlay_stops - 1) / 2
        overlay_colors = ("#74BFE5", "#318FCA", "#1556A3", "#061C3F")
        for n, color, values in zip(sample_overlay_sizes, overlay_colors, overlay_grouped):
            occupied = np.flatnonzero(values)
            sample_overlay_axis.bar(
                overlay_centers[occupied], values[occupied] / beta2_n_sweep_M,
                width=.94 * (overlay_stops - overlay_starts)[occupied],
                color=color, alpha=.30, edgecolor="white", linewidth=.3,
                label="_nolegend_",
            )
        from matplotlib import patheffects as overlay_path_effects
        # Draw all connecting lines after the fills so each curve remains visible.
        for n, color, values in zip(sample_overlay_sizes, overlay_colors, overlay_grouped):
            occupied = np.flatnonzero(values)
            first = max(0, occupied[0] - 1)
            last = min(len(values), occupied[-1] + 2)
            sample_overlay_axis.plot(
                overlay_centers[first:last], values[first:last] / beta2_n_sweep_M,
                color=color, linestyle="-", linewidth=1.9, label=rf"$n={n}$", zorder=4,
                path_effects=[
                    overlay_path_effects.Stroke(linewidth=3.1, foreground="white"),
                    overlay_path_effects.Normal(),
                ],
            )
        occupied_bins = np.flatnonzero(overlay_grouped.sum(axis=0))
        overlay_left = max(-.5, overlay_starts[occupied_bins[0]] - 2 * beta2_n_sweep_binning - .5)
        overlay_right = min(beta2_n_sweep_N - .5,
                            overlay_stops[occupied_bins[-1]] + 2 * beta2_n_sweep_binning - .5)
        overlay_h_ticks = np.array([.05, .06, .08, .1, .15, .2, .25, .3, .4, .5, .6])
        overlay_tick_positions = np.interp(
            np.log(overlay_h_ticks), np.log(beta2_n_sweep_h_grid), np.arange(beta2_n_sweep_N),
        )
        visible = (overlay_tick_positions >= overlay_left) & (overlay_tick_positions <= overlay_right)
        sample_overlay_axis.set_xticks(
            overlay_tick_positions[visible], [f"{h:g}" for h in overlay_h_ticks[visible]],
        )
        sample_overlay_axis.set_xlim(overlay_left, overlay_right)
        sample_overlay_axis.set_ylim(0, 1.10 * overlay_grouped.max() / beta2_n_sweep_M)
        from matplotlib.ticker import MultipleLocator, PercentFormatter
        sample_overlay_axis.yaxis.set_major_locator(MultipleLocator(tick_step))
        sample_overlay_axis.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
        sample_overlay_axis.set_xlabel("selected bandwidth h", fontsize=12, labelpad=12, color="#304967")
        sample_overlay_axis.set_ylabel("relative frequency", fontsize=12, labelpad=12, color="#304967")
        sample_overlay_axis.tick_params(labelsize=10, colors="#304967", length=0)
        sample_overlay_axis.grid(axis="y", color="#E5ECF6", linewidth=.6)
        sample_overlay_axis.set_axisbelow(True)
        for side in ("top", "right", "left"):
            sample_overlay_axis.spines[side].set_visible(False)
        sample_overlay_axis.spines["bottom"].set_color("#E5ECF6")
        sample_overlay_axis.legend(loc="upper center", bbox_to_anchor=(.5, -.20),
                                      ncol=4, frameon=False, fontsize=11, labelcolor="#304967")
        sample_overlay_figure.suptitle(
            plot_title + "\n"
            + rf"$N={beta2_n_sweep_N}$, $M={beta2_n_sweep_M}$, ${lambda_text}$; "
            + f"{beta2_n_sweep_binning} grid positions per bin",
            y=.965, fontsize=15, color="#304967",
        )
        sample_overlay_pdf = project_root / "output/pdf" / output_filename
        sample_overlay_pdf.parent.mkdir(parents=True, exist_ok=True)
        sample_overlay_figure.savefig(sample_overlay_pdf, format="pdf", bbox_inches="tight")
        np.savetxt(
            beta2_n_sweep_output_directory / csv_filename,
            np.column_stack((overlay_starts, overlay_stops - 1, overlay_grouped.T)),
            delimiter=",", fmt="%d",
            header="first_grid_index,last_grid_index,n300,n400,n500,n1000", comments="",
        )
        plt.show()  # Shared automatic exporter also saves the new overlay as PDF.
        plt.close(sample_overlay_figure)
        return {"figure": sample_overlay_figure, "axis": sample_overlay_axis,
                "counts": overlay_grouped, "pdf": sample_overlay_pdf}


localized_overlay_plot = plot_spectral_sample_size_overlay(
    count_key="localized_counts",
    plot_title="Localized spectral bandwidth distributions across sample sizes",
    lambda_text=r"\lambda_N=N^{3/2}",
    output_filename="localized_spectral_bandwidths_across_n_N1000.pdf",
    csv_filename="localized_overlay_histogram_counts.csv", tick_step=.01,
)

lambda1_overlay_plot = plot_spectral_sample_size_overlay(
    count_key="counts",
    plot_title="Spectral bandwidth distributions across sample sizes",
    lambda_text=r"\lambda=1",
    output_filename="spectral_bandwidths_across_n_lambda1_N1000.pdf",
    csv_filename="lambda1_overlay_histogram_counts.csv", tick_step=.05,
)
